# SHJ Learning Dynamics: Human vs Neural Models

This notebook analyzes the results of the SHJ category learning experiment, comparing human performance against two neural network models (Logistic Regression and MLP).

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Set plot style
sns.set_context('notebook', font_scale=1.2)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
%matplotlib inline

## 1. Load Data

We'll load the model training results and the collected human participant data.

In [ ]:
# Paths
MODEL_RESULTS_PATH = Path('../results/model_results.json')
HUMAN_DATA_DIR = Path('../experiment/data/')

# 1. Load Model Results
try:
    with open(MODEL_RESULTS_PATH, 'r') as f:
        model_results = json.load(f)
    print("✅ Validating Model Results:")
    print(f"   Found models: {list(model_results.keys())}")
except FileNotFoundError:
    print("❌ Model results not found! Run 'python models/train.py' first.")
    model_results = {}

# 2. Load Human Data
human_files = list(HUMAN_DATA_DIR.glob('*.json'))
human_data = []

print(f"\n✅ Loading Human Data:")
print(f"   Found {len(human_files)} data files")

for p in human_files:
    try:
        with open(p, 'r') as f:
            data = json.load(f)
            human_data.append(data)
            print(f"   - Loaded {data.get('participant_id', 'Unknown')} ({len(data.get('trials', []))} trials)")
    except Exception as e:
        print(f"   ❌ Error loading {p.name}: {e}")

## 2. Process Data

We need to process the raw human trial data into learning curves and trials-to-criterion stats to match the model format.

In [ ]:
def process_human_learning_curves(human_data, max_trials=160, window_size=20, criterion=16):
    processed = {
        'type_1': {'curves': [], 'ttc': []},
        'type_2': {'curves': [], 'ttc': []},
        'type_6': {'curves': [], 'ttc': []}
    }
    
    if not human_data:
        return processed

    for participant in human_data:
        # Extract trials for this participant
        p_trials = participant.get('trials', [])
        
        # Separate by SHJ type
        type_trials = {}
        for t in p_trials:
            ts = t['shj_type']
            if ts not in type_trials:
                type_trials[ts] = []
            type_trials[ts].append(t['accuracy'])
        
        # Calculate metrics for each type this participant completed
        for shj_type, accs in type_trials.items():
            if shj_type not in processed:
                continue
                
            # 1. Learning Curve (pad with final value like models)
            curve = accs.copy()
            if len(curve) < max_trials:
                curve += [curve[-1]] * (max_trials - len(curve))
            elif len(curve) > max_trials:
                curve = curve[:max_trials]
            
            # 2. Trials to Criterion
            # Re-calculate based on rolling window to be sure, or use trial length if stopped by criterion
            ttc = None
            # Check if they actually reached criterion in the data
            # (The experiment code stops them, so len(accs) is usually TTC if successful)
            # We'll check the last window
            if len(accs) >= window_size:
                last_window = accs[-window_size:]
                if sum(last_window) >= criterion:
                    ttc = len(accs)
            
            # If they maxed out trials and didn't reach criterion, TTC is usually considered max_trials or undefined
            # For visualization, we often cap it at max_trials
            if ttc is None and len(accs) >= max_trials:
                ttc = max_trials # or None to exclude
            
            if ttc:
                processed[shj_type]['ttc'].append(ttc)
            
            processed[shj_type]['curves'].append(curve)
            
    return processed

human_results = process_human_learning_curves(human_data)

## 3. Visualization: Learning Curves

Comparing average accuracy over trials for Human vs. Neural Models.

In [ ]:
def plot_comparison_curves(shj_type, title):
    plt.figure(figsize=(10, 6))
    
    # 1. Plot Human Data
    h_data = human_results.get(shj_type)
    if h_data and h_data['curves']:
        curves = np.array(h_data['curves'])
        # Calculate mean and CI
        mean = curves.mean(axis=0)
        sem = curves.std(axis=0) / np.sqrt(len(curves))
        ci = 1.96 * sem
        trials = np.arange(1, len(mean) + 1)
        
        # Apply smoothing for readability if needed, but raw is better for scientific accuracy
        # We'll use a rolling mean for the plot line but raw for fill
        series = pd.Series(mean)
        smooth_mean = series.rolling(window=5, min_periods=1).mean()
        
        plt.plot(trials, smooth_mean, color='black', linewidth=3, label=f'Humans (N={len(curves)})')
        plt.fill_between(trials, mean-ci, mean+ci, color='black', alpha=0.15)
        
    # 2. Plot Model Data
    colors = {'logistic': 'blue', 'mlp': 'red'}
    labels = {'logistic': 'Logistic Regression', 'mlp': 'Small MLP'}
    
    for model_name, res in model_results.items():
        if shj_type in res:
            curve_data = res[shj_type]['learning_curve']
            mean = np.array(curve_data['mean'])
            std = np.array(curve_data['std'])
            trials = np.array(curve_data['trials'])
            
            plt.plot(trials, mean, color=colors[model_name], label=labels[model_name], linewidth=2)
            plt.fill_between(trials, mean-std, mean+std, color=colors[model_name], alpha=0.1)
            
    plt.title(f'Learning Dynamics: {title}', fontsize=16, fontweight='bold')
    plt.xlabel('Trial Number', fontsize=14)
    plt.ylabel('Accuracy', fontsize=14)
    plt.ylim([0, 1.05])
    plt.axhline(0.5, linestyle='--', color='gray', alpha=0.5, label='Chance')
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save
    save_path = Path(f'../results/figures/curve_{shj_type}.png')
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=300)
    plt.show()

# Generate plots
shj_names = {
    'type_1': 'Type I (Single Feature)',
    'type_2': 'Type II (XOR)',
    'type_6': 'Type VI (Family Resemblance)'
}

for t, name in shj_names.items():
    print(f"Plotting {name}...")
    plot_comparison_curves(t, name)

## 4. Analysis: Difficulty Ordering

We compare the median "Trials to Criterion" (TTC) to determine the difficulty ordering for each system.

**Human Expected Ordering**: Type I < Type II < Type VI  
**Neural Expected Ordering**: Often Type I < Type VI < Type II (XOR is hard for linear/shallow models)

In [ ]:
# Compile Summary Data
rows = []

# Human stats
for shj_type in ['type_1', 'type_2', 'type_6']:
    ttcs = human_results[shj_type]['ttc']
    if ttcs:
        rows.append({
            'System': 'Humans',
            'SHJ Type': shj_names[shj_type],
            'Median TTC': np.median(ttcs),
            'N': len(ttcs)
        })
    else:
         rows.append({
            'System': 'Humans',
            'SHJ Type': shj_names[shj_type],
            'Median TTC': np.nan,
            'N': 0
        })

# Model stats
for model_name in ['logistic', 'mlp']:
    sys_name = 'Logistic Regression' if model_name == 'logistic' else 'MLP'
    for shj_type in ['type_1', 'type_2', 'type_6']:
        # Use model reported median
        median_ttc = model_results[model_name][shj_type]['trials_to_criterion']['median']
        # If None (didn't finish), use max
        if median_ttc is None:
            median_ttc = 160
            
        rows.append({
            'System': sys_name,
            'SHJ Type': shj_names[shj_type],
            'Median TTC': median_ttc,
            'N': 10
        })

df_summary = pd.DataFrame(rows)

# Plot Difficulty Ordering
plt.figure(figsize=(10, 6))
sns.barplot(data=df_summary, x='SHJ Type', y='Median TTC', hue='System', palette=['black', 'blue', 'red'])

plt.title('Difficulty Ordering Comparison', fontsize=16, fontweight='bold')
plt.ylabel('Median Trials to Criterion (Lower = Easier)', fontsize=14)
plt.xlabel('')
plt.xticks(rotation=15)
plt.tight_layout()

plt.savefig('../results/figures/difficulty_ordering.png', dpi=300)
plt.show()

# Display Table
display(df_summary)